# ASID Data Cleaning Pipeline
## Australian Shark Incident Database — Preparation for Geospatial Visualisation

**Project:** Australian Shark Incident Database Interactive Map  
**Author:** Warren van Ryn  
**Data source:** [ASID · Flinders University](https://github.com/cjabradshaw/AustralianSharkIncidentDatabase)  
**Output:** `asid_cleaned.csv` — consumed by a MapLibre GL JS / D3 interactive dashboard

---

### What this notebook does

The ASID public release is an Excel workbook with 1,296 incident records and 60 fields spanning 1791 to the present. Before geospatial rendering, several data quality issues require systematic treatment:

| Issue | Records affected | Treatment |
|---|---|---|
| Species name drift (synonyms, taxonomic revision) | ~930 | Dictionary-based standardisation |
| `bronze whaler shark` missing from synonym list | 32 | Patch dictionary before applying |
| Categorical field casing inconsistency | 9 | Lowercase normalisation |
| Longitude data entry error (Forbes Island: 4034) | 1 | Corrected to 143.40 |
| Low-precision coordinates (≤ 2 decimal places) | 19 | Flagged `coord_quality = 'low'` |
| Null provoked/unprovoked values | 6 | Set to `'unknown'` |
| Uncovered species groups (wobbegong, whaler shark…) | ~315 | Supplementary dictionary entries |

The notebook walks through each step with inline commentary, then writes a minified columnar CSV ready for direct consumption by the front-end.

### Dependencies
```
pip install pandas openpyxl
```

---
## 1. Imports and File Paths

In [ ]:
import pandas as pd
import numpy as np

# ── Input files ────────────────────────────────────────────────────────────────
ASID_FILE = "Australian_Shark-Incident_Database_Public_Version.xlsx"
DICT_FILE  = "australian_shark_species_dictionary.csv"

# ── Output file ────────────────────────────────────────────────────────────────
OUTPUT_FILE = "data/asid_cleaned.csv"

# Change log — populated as the pipeline runs
changes = {k: 0 for k in [
    "state_normalised", "injury_normalised", "provoked_normalised",
    "sci_name_standardised", "common_name_standardised",
    "coord_corrected", "coord_flagged_low_precision"
]}

---
## 2. Load Source Data

The ASID workbook uses a trailing empty column (`Unnamed: 59`) which we drop on load.  
The species dictionary CSV defines 16 canonical species with comma-separated synonym lists
for both scientific and common names.

In [ ]:
df  = pd.read_excel(ASID_FILE)
dic = pd.read_csv(DICT_FILE)

# Drop trailing artefact column
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

print(f"ASID records loaded:        {len(df):,}")
print(f"ASID columns:               {len(df.columns)}")
print(f"Dictionary entries (base):  {len(dic)}")
print(f"Year range:                 {df['Incident.year'].min()} – {df['Incident.year'].max()}")

ASID records loaded:        1,296
ASID columns:               59
Dictionary entries (base):  16
Year range:                 1791 – 2026


---
## 3. Extend the Species Dictionary

The base dictionary omits four species/groups that appear in significant numbers in the ASID.
These are added programmatically rather than editing the CSV, so the extension logic is
version-controlled and auditable.

**Design decisions:**

- **Wobbegong** (215 records): Multiple *Orectolobus* spp. are grouped under this label
  in the ASID. Resolving to species level is impossible without per-record review.
  Scientific placeholder: `Orectolobus spp.`

- **Whaler shark** (79 records): Historical catch-all for multiple *Carcharhinus* spp.
  Not resolvable. Kept as group label with `Carcharhinus spp.`

- **Hammerhead shark** (5 records): Group label → `Sphyrna spp.`

- **Shortfin mako** (2 records): Identifiable to species → `Isurus oxyrinchus`

Additionally, the base dictionary entry for *Bronze Whaler* is missing `bronze whaler shark`
(with the "shark" suffix) from its common name synonyms, causing 32 records to miss on
common-name matching while still hitting on scientific name. This is patched below.

In [ ]:
supplementary = [
    {
        "canonical_scientific_name": "Orectolobus spp.",
        "scientific_synonyms":       "Orectolobus maculatus, Orectolobus ornatus",
        "canonical_common_name":     "wobbegong",
        "common_name_synonyms":      "wobbegong shark, carpet shark",
        "notes": "Group-level; multiple Orectolobus species recorded under this label"
    },
    {
        "canonical_scientific_name": "Carcharhinus spp.",
        "scientific_synonyms":       "",
        "canonical_common_name":     "whaler shark",
        "common_name_synonyms":      "whaler, requiem shark",
        "notes": "Group-level; cannot be resolved to single species"
    },
    {
        "canonical_scientific_name": "Sphyrna spp.",
        "scientific_synonyms":       "Sphyrna mokarran, Sphyrna zygaena",
        "canonical_common_name":     "hammerhead shark",
        "common_name_synonyms":      "hammerhead, great hammerhead, smooth hammerhead",
        "notes": "Group-level"
    },
    {
        "canonical_scientific_name": "Isurus oxyrinchus",
        "scientific_synonyms":       "Isurus glaucus",
        "canonical_common_name":     "shortfin mako",
        "common_name_synonyms":      "shortfin mako shark, mako shark, blue pointer",
        "notes": "Species-level"
    },
]
dic = pd.concat([dic, pd.DataFrame(supplementary)], ignore_index=True)

# Patch: add 'bronze whaler shark' to the existing Bronze Whaler synonyms
bw_mask = dic["canonical_common_name"].str.lower() == "bronze whaler"
dic.loc[bw_mask, "common_name_synonyms"] = (
    dic.loc[bw_mask, "common_name_synonyms"].fillna("") +
    ", bronze whaler shark, bronzie, copper shark"
)

print(f"Dictionary extended to {len(dic)} entries")
print(f"Bronze Whaler synonyms: {dic.loc[bw_mask, 'common_name_synonyms'].values[0]}")

Dictionary extended to 20 entries
Bronze Whaler synonyms:  Copper Shark, Narrowtooth Shark, Bronzie, bronze whaler shark, bronzie, copper shark


---
## 4. Build Synonym Lookup Tables

Two flat dictionaries are constructed — `sci_lookup` and `com_lookup` — mapping any known
synonym (lower-cased) to its canonical form. These allow O(1) lookup per record rather than
iterating over the dictionary for every row.

In [ ]:
sci_lookup = {}
com_lookup = {}

for _, row in dic.iterrows():
    canon_sci = str(row["canonical_scientific_name"]).strip()
    canon_com = str(row["canonical_common_name"]).strip()

    sci_syns = [s.strip() for s in str(row["scientific_synonyms"]).split(",") if s.strip()]
    sci_syns.append(canon_sci)
    for s in sci_syns:
        sci_lookup[s.lower()] = canon_sci

    com_syns = [s.strip() for s in str(row["common_name_synonyms"]).split(",") if s.strip()]
    com_syns.append(canon_com)
    for s in com_syns:
        com_lookup[s.lower()] = canon_com

print(f"Scientific name synonyms indexed:  {len(sci_lookup)}")
print(f"Common name synonyms indexed:      {len(com_lookup)}")

Scientific name synonyms indexed:  53
Common name synonyms indexed:      61


---
## 5. Categorical Field Normalisation

Three fields contain casing inconsistencies. Rule: strip whitespace, lowercase,
then apply field-specific canonical mappings.

| Field | Issue | Fix |
|---|---|---|
| `State` | `'Qld'` instead of `'QLD'` | Map via `state_map` dict |
| `Victim.injury` | `'Injured'`, `'injury'` | Lowercase + `injury_map` |
| `Provoked/unprovoked` | 6 null values | Set to `'unknown'` |

In [ ]:
# ── State ─────────────────────────────────────────────────────────────────────
state_map = {"qld": "QLD"}

for idx, val in df["State"].items():
    if pd.isna(val): continue
    norm = state_map.get(str(val).strip().lower(), str(val).strip().upper())
    if norm != str(val).strip():
        print(f"  [State] row {idx}: '{val}' → '{norm}'")
        df.at[idx, "State"] = norm
        changes["state_normalised"] += 1

# ── Victim.injury ──────────────────────────────────────────────────────────────
# 'injury' (1 record) is treated as 'injured' — best available interpretation
injury_map = {"injured":"injured","fatal":"fatal","uninjured":"uninjured",
              "unknown":"unknown","injury":"injured"}

for idx, val in df["Victim.injury"].items():
    if pd.isna(val): continue
    norm = injury_map.get(str(val).strip().lower(), str(val).strip().lower())
    if norm != str(val).strip():
        print(f"  [Victim.injury] row {idx}: '{val}' → '{norm}'")
        df.at[idx, "Victim.injury"] = norm
        changes["injury_normalised"] += 1

# ── Provoked/unprovoked ────────────────────────────────────────────────────────
null_count = 0
for idx, val in df["Provoked/unprovoked"].items():
    if pd.isna(val):
        df.at[idx, "Provoked/unprovoked"] = "unknown"
        null_count += 1
        changes["provoked_normalised"] += 1

print(f"  [Provoked/unprovoked] {null_count} null values → 'unknown'")
print(f"\nCategorical normalisation complete.")

  [State] row 1288: 'Qld' → 'QLD'
  [Victim.injury] row 975: 'Injured' → 'injured'
  [Victim.injury] row 1220: 'injury' → 'injured'
  [Provoked/unprovoked] 6 null values → 'unknown'

Categorical normalisation complete.


---
## 6. Species Name Standardisation

For each record, the scientific name is checked first against `sci_lookup`.
The common name is checked independently against `com_lookup`.
Both checks run on every record — a match on scientific name does not skip
the common name check, since historical records sometimes have one field but
not the other populated correctly.

In [ ]:
sci_hits = com_hits = 0

for idx, row in df.iterrows():
    sci_raw = str(row["Shark.scientific.name"]).strip() if pd.notna(row["Shark.scientific.name"]) else ""
    com_raw = str(row["Shark.common.name"]).strip()     if pd.notna(row["Shark.common.name"])     else ""

    if sci_raw.lower() in sci_lookup:
        canon = sci_lookup[sci_raw.lower()]
        if canon != sci_raw:
            df.at[idx, "Shark.scientific.name"] = canon
            sci_hits += 1
            changes["sci_name_standardised"] += 1

    if com_raw.lower() in com_lookup:
        canon = com_lookup[com_raw.lower()]
        if canon != com_raw:
            df.at[idx, "Shark.common.name"] = canon
            com_hits += 1
            changes["common_name_standardised"] += 1

print(f"Scientific names standardised: {sci_hits}")
print(f"Common names standardised:     {com_hits}")
print(f"\nTop common names post-standardisation:")
print(df["Shark.common.name"].value_counts().head(8).to_string())

Scientific names standardised: 0
Common names standardised:     930

Top common names post-standardisation:
Shark.common.name
White Shark      386
Tiger Shark      236
Bull Shark       218
wobbegong        215
whaler shark      79
Bronze Whaler     32
unknown           18
Grey Nurse Shark  10


---
## 7. Coordinate Cleaning and Precision Flagging

### 7a. Known data entry error

Row 347 (Forbes Island, QLD) has `Longitude = 4034`. The latitude `−12.2929` is
precise to 4 decimal places and consistent with Forbes Island's actual position.
The longitude is a clear transposition of `143.4` (digit order scrambled: 1-4-3-4 → 4-0-3-4).
Corrected to `143.40`; flagged in a `coord_corrected` boolean column.

### 7b. Coordinate precision flagging

Records with ≤ 2 decimal places of latitude precision were assigned from a town centroid
or rough map click rather than GPS. At 1 d.p. that is ±11 km uncertainty; at 2 d.p. ±1 km.
These 19 records render as **hollow markers** in the visualisation so viewers can
distinguish approximate from precise locations.

This is a cartographic transparency measure, not a data removal decision — the records
remain in the dataset and contribute to all counts and charts.

In [ ]:
df["Latitude"]  = pd.to_numeric(df["Latitude"],  errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

# ── 7a. Forbes Island longitude fix ───────────────────────────────────────────
forbes = (df["Location"].str.lower().str.contains("forbes island", na=False)) &          (df["Longitude"] > 1000)

if forbes.any():
    idx = df[forbes].index[0]
    old_lon = df.at[idx, "Longitude"]
    df.at[idx, "Longitude"] = 143.40
    print(f"[COORD FIX] row {idx}: Longitude {old_lon} → 143.40")
    print(f"            (Forbes Island, QLD — transposition error)")
    changes["coord_corrected"] += 1

df["coord_corrected"] = False
df.loc[forbes, "coord_corrected"] = True

# ── 7b. Precision flagging ─────────────────────────────────────────────────────
def decimal_places(val):
    if pd.isna(val): return 0
    s = f"{val:.10f}".rstrip("0")
    return len(s.split(".")[1]) if "." in s else 0

dp = df["Latitude"].apply(decimal_places)
low_prec = dp <= 2
df["coord_quality"] = np.where(low_prec, "low", "high")

changes["coord_flagged_low_precision"] = int(low_prec.sum())
print(f"\n[PRECISION] {low_prec.sum()} records flagged as low-precision (≤ 2 d.p.)")
print(f"            These will render as hollow markers in the visualisation.")
print(f"\nLow-precision records:")
print(df[low_prec][["Location","Latitude","Longitude","coord_quality"]].to_string())

[COORD FIX] row 347: Longitude 4034.0 → 143.40
            (Forbes Island, QLD — transposition error)

[PRECISION] 19 records flagged as low-precision (≤ 2 d.p.)
            These will render as hollow markers in the visualisation.

Low-precision records:
                                         Location  Latitude  Longitude coord_quality
0                               sydney (near)    -33.86    151.20           low
15                              moreton bay      -27.25    153.25           low
22                           sydney harbour      -33.85    151.22           low
50       torres strait, unknown pearl beds      -10.45    142.39           low
53             port walcott & de grey river      -20.70    117.25           low
55       torres strait, unknown pearl beds      -10.45    142.39           low
124                          melville island      -11.60    130.95           low
126      torres strait, unknown pearl beds      -10.45    142.39           low
128                  

---
## 8. Write Output and Verify

The output CSV adds two columns to the original data:
- `coord_quality`: `'high'` or `'low'` — used by the front-end to toggle marker style
- `coord_corrected`: boolean — flags the single manually corrected record

All other columns are preserved. The file is written to `data/` to match the
expected path in the MapLibre front-end (`./data/asid_cleaned.csv`).

In [ ]:
import os
os.makedirs("data", exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)
print(f"Written: {OUTPUT_FILE}")
print(f"Records: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print(f"\n{'='*50}")
print("CLEANING SUMMARY")
print(f"{'='*50}")
for k, v in changes.items():
    print(f"  {k:<42} {v:>4} records")
print(f"{'='*50}")

Written: data/asid_cleaned.csv
Records: 1,296
Columns: 61

CLEANING SUMMARY
  state_normalised                                1 records
  injury_normalised                               2 records
  provoked_normalised                             6 records
  sci_name_standardised                           0 records
  common_name_standardised                      930 records
  coord_corrected                                 1 records
  coord_flagged_low_precision                    19 records


---
## 9. Output Verification

In [ ]:
cleaned = pd.read_csv(OUTPUT_FILE)

print("coord_quality distribution:")
print(cleaned["coord_quality"].value_counts().to_string())

print("\nVictim.injury (post-clean):")
print(cleaned["Victim.injury"].value_counts().to_string())

print("\nState (post-clean):")
print(cleaned["State"].value_counts().to_string())

print("\nRecords with null lat/lon:")
null_lat = cleaned["Latitude"].isna().sum()
null_lon = cleaned["Longitude"].isna().sum()
print(f"  Null latitude:  {null_lat}  (these will be excluded from map rendering)")
print(f"  Null longitude: {null_lon}")

coord_quality distribution:
coord_quality
high    1277
low       19
Name: count, dtype: int64

Victim.injury (post-clean):
Victim.injury
injured      782
fatal        262
uninjured    248
unknown        4
Name: count, dtype: int64

State (post-clean):
State
NSW    470
QLD    377
WA     243
SA      85
VIC     73
TAS     28
NT      19
Name: count, dtype: int64

Records with null lat/lon:
  Null latitude:  2  (these will be excluded from map rendering)
  Null longitude: 0


---
## 10. Front-End Implementation Notes

The cleaned CSV feeds directly into an `index.html` built with **MapLibre GL JS** and **D3**.
Key design decisions in the front-end:

### Species colour coding
Five species are assigned distinct colours chosen for contrast against the ESRI Ocean
basemap (blue tones); all others fall back to grey:

| Species | Colour | Hex |
|---|---|---|
| White Shark | Coral red | `#ff6b6b` |
| Tiger Shark | Warm amber | `#ffa94d` |
| Bull Shark | Sky blue | `#74c0fc` |
| Wobbegong | Lime green | `#a9e34b` |
| Bronze Whaler | Purple | `#da77f2` |
| Other / Unknown | Grey | `#868e96` |

### Coordinate precision markers
Records with `coord_quality == 'low'` render as **hollow circle markers** (transparent fill,
coloured stroke) so positional uncertainty is visually apparent without hiding the incident.

### Clustering
MapLibre GL's built-in GeoJSON clustering groups nearby incidents at zoom levels ≤ 8.
Cluster circles are sized by count (16 / 22 / 30 px radius) and use a density-coded
colour scheme (amber → orange → red).

### Linked D3 charts
Three SVG charts in the sidebar update reactively with filter state:
- Monthly bar chart (seasonal incident distribution)
- Species horizontal bar chart (top 6 + Other, species-coloured)
- State stacked bar chart (incident count by outcome per state)

### Basemap
ESRI World Ocean Base (`server.arcgisonline.com`) with the Ocean Reference label overlay,
providing GEBCO-derived bathymetry, continental shelf detail, and named ocean features
relevant to shark ecology (estuaries, reef areas, river mouths).

### Coordinate note
Two records have null latitude and are silently excluded from map rendering via:
```javascript
.filter(d => d.lat !== null && d.lon !== null)
```
They remain in all sidebar chart counts since they have valid state and species data.